In [1]:
!pip install transformers datasets torch pandas gradio accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import gradio as gr
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch
import warnings
import os

warnings.filterwarnings("ignore")

def setup_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    os.system("pip install transformers datasets torch pandas gradio accelerate")

setup_colab()

Mounted at /content/drive


In [3]:
model_name = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to("cuda")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
def load_and_preprocess_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset not found at {file_path}. Please check the path and try again.")

    # Read CSV with latin-1 encoding to handle non-UTF-8 characters
    df = pd.read_csv(file_path, encoding='latin-1')
    required_columns = ["body_part", "medical_condition", "symptoms", "drug_name", "side_effects"]
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"Dataset must contain columns: {required_columns}")

    df = df[required_columns]
    df["symptoms"] = df["symptoms"].astype(str).fillna("")
    alt_drugs = df.groupby("medical_condition")["drug_name"].apply(list).to_dict()

    def create_prompt(row):
        input_text = f"Body part: {row['body_part']}, Symptoms: {row['symptoms']}"
        output_text = (f"Condition: {row['medical_condition']}, Drug: {row['drug_name']}, "
                      f"Side effects: {row['side_effects']}")
        return {"input": input_text, "output": output_text, "condition": row["medical_condition"]}

    data = df.apply(create_prompt, axis=1).tolist()
    dataset = Dataset.from_list(data)

    def tokenize_function(examples):
        inputs = tokenizer(examples["input"], padding="max_length", truncation=True, max_length=128)
        outputs = tokenizer(examples["output"], padding="max_length", truncation=True, max_length=128)
        inputs["labels"] = outputs["input_ids"]
        return inputs

    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    return tokenized_dataset, df, alt_drugs

dataset_path = "/content/drive/MyDrive/HPC/last_dataset_hpc.csv"  # Your dataset path
tokenized_dataset, df, alt_drugs = load_and_preprocess_data(dataset_path)

Map:   0%|          | 0/463 [00:00<?, ? examples/s]

In [5]:
import os
os.environ["WANDB_MODE"] = "disabled"  # Disable W&B logging

def fine_tune_model(tokenized_dataset):
    training_args = TrainingArguments(
        output_dir="./flan-t5-medical",
        run_name="flan-t5-medical-run",  # Unique run name
        per_device_train_batch_size=8,
        num_train_epochs=3,
        save_steps=500,
        save_total_limit=2,
        fp16=True,  # Mixed precision for GPU efficiency
        logging_dir="./logs",
        logging_steps=100,
        report_to="none",  # Disable all external logging (W&B, TensorBoard)
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
    )
    trainer.train()

    # Save fine-tuned model to Google Drive
    model.save_pretrained("/content/drive/MyDrive/flan-t5-medical")
    tokenizer.save_pretrained("/content/drive/MyDrive/flan-t5-medical")

# Clear GPU memory before training
torch.cuda.empty_cache()

fine_tune_model(tokenized_dataset)

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
100,0.000000


Intermediate Result

In [6]:
import re
import gradio as gr
import pandas as pd
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer

# Global state for storing available drugs for current prediction
current_drug_list = []

# Load Qwen model and tokenizer for conversational module
qwen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_model_name, torch_dtype=torch.float16).to("cuda")
qwen_model.eval()

def predict_condition_and_drug(body_part, symptoms):
    global current_drug_list

    if isinstance(symptoms, str):
        symptoms = [symptoms]

    input_text = f"Body part: {body_part}, Symptoms: {', '.join(symptoms)}"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_length=100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Try extracting condition from model output
    condition_match = re.search(r"(Condition|condition)\s*[:=]\s*([A-Za-z0-9 \-]+)", response, re.IGNORECASE)
    condition = condition_match.group(2).strip() if condition_match else None

    if not condition:
        # Fallback: Match symptoms with df using overlap
        df_filtered = df[df["body_part"] == body_part]
        condition_scores = {}

        for _, row in df_filtered.iterrows():
            row_symptoms = row["symptoms"]
            if pd.isna(row_symptoms):
                continue
            row_symptom_set = set(row_symptoms.lower().split(", "))
            overlap = set(s.lower() for s in symptoms) & row_symptom_set
            score = len(overlap)
            if score > 0:
                condition_scores[row["medical_condition"]] = condition_scores.get(row["medical_condition"], 0) + score

        if condition_scores:
            condition = max(condition_scores.items(), key=lambda x: x[1])[0]
        else:
            condition = "Unknown"

    # Drug suggestion logic
    drug = "Unknown"
    side_effects = "Not listed for this drug"
    alt_drug = "None available"
    current_drug_list = []

    if condition and condition != "Unknown":
        condition_df = df[df["medical_condition"].str.lower() == condition.lower()]
        condition_drugs = condition_df["drug_name"].dropna().unique().tolist()
        current_drug_list = condition_drugs

        # Extract model-suggested drug
        try:
            model_drug = re.search(r"(Drug|drug)\s*[:=]\s*([A-Za-z0-9 \-]+)", response, re.IGNORECASE)
            model_drug = model_drug.group(2).strip() if model_drug else ""
        except:
            model_drug = ""

        drug = model_drug if model_drug in condition_drugs else (condition_drugs[0] if condition_drugs else "Unknown")

        # Get side effects from dataset for the primary drug
        if drug != "Unknown" and drug != "No drug found":
            drug_match = condition_df[condition_df["drug_name"].str.lower() == drug.lower()]
            if not drug_match.empty and not pd.isna(drug_match["side_effects"].iloc[0]):
                side_effects = drug_match["side_effects"].iloc[0]
            else:
                side_effects = "Not listed for this drug"

        alt_candidates = [d for d in condition_drugs if d != drug]
        alt_drug = alt_candidates[0] if alt_candidates else "None available"
    else:
        drug = "No drug found"

    return condition, drug, alt_drug, side_effects

def conversational_medical_chat(user_input, chat_history):
    chat_history = chat_history or []

    if not user_input.strip():
        chat_history.append(("Bot", "Please enter a medical question or describe your symptoms."))
        return chat_history, ""

    # Append user input to chat history
    chat_history.append(("User", user_input))

    # Prepare prompt for Qwen model with medical context
    prompt = (
        "You are a medical assistant chatbot specializing in providing accurate and concise information about medical conditions, symptoms, and treatments. "
        "Answer the user's question in a clear, professional, and empathetic manner. If the query is vague, ask for clarification. "
        "Use the provided dataset context if relevant, but do not disclose dataset details. "
        f"User query: {user_input}"
    )

    # Tokenize and generate response
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = qwen_model.generate(
        **inputs,
        max_length=500,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=qwen_tokenizer.pad_token_id,
        eos_token_id=qwen_tokenizer.eos_token_id
    )
    response = qwen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean response (remove prompt part if included)
    response = response.replace(prompt, "").strip()

    # Append bot response to chat history
    chat_history.append(("Bot", response))
    return chat_history, ""

def chatbot_interface(df):
    body_parts = df["body_part"].dropna().unique().tolist()
    body_parts.sort()

    def update_symptoms(selected_body_part):
        symptoms_list = df[df["body_part"] == selected_body_part]["symptoms"].str.split(", ").explode().dropna().unique().tolist()
        symptoms_list.sort()
        return gr.CheckboxGroup(choices=symptoms_list, value=[], label="Select Symptoms")

    def submit(body_part, symptoms, chat_history):
        chat_history = chat_history or []

        if not body_part or not symptoms:
            chat_history.append(("Bot", "Please select a body part and at least one symptom, then click Submit."))
            return chat_history, symptoms

        # Validate symptoms against available choices
        symptoms_list = df[df["body_part"] == body_part]["symptoms"].str.split(", ").explode().dropna().unique().tolist()
        if not all(s in symptoms_list for s in symptoms):
            chat_history.append(("Bot", "One or more selected symptoms are invalid. Please choose symptoms from the list and try again."))
            return chat_history, symptoms

        # Append user input (body part and symptoms) to chat history
        user_message = f"**Selected Body Part**: {body_part}\n**Selected Symptoms**: {', '.join(symptoms)}"
        chat_history.append(("User", user_message))

        # Generate bot response
        condition, drug, alt_drug, side_effects = predict_condition_and_drug(body_part, symptoms)
        bot_response = (
            f"**Predicted Medical Condition**: {condition}\n\n"
            f"**Suggested Drug**: {drug}\n\n"
            f"**Alternative Drug (if primary is unavailable)**: {alt_drug}\n\n"
            f"**Possible Side Effects of {drug}**: {side_effects}\n\n"
            "To start a new scan, select a new body part and symptoms, then click Submit."
        )
        chat_history.append(("Bot", bot_response))
        return chat_history, []  # Clear symptoms for new scan

    with gr.Blocks(theme=gr.themes.Soft()) as interface:
        gr.Markdown("### 🧠 SymptoScan (By Team-50 from HPC)")
        gr.Markdown("Explore medical condition predictions or ask free-form medical questions.")

        with gr.Tabs():
            with gr.TabItem("Symptom Scanner"):
                gr.Markdown("Select a body part and related symptoms to predict a medical condition, receive medication suggestions, and view possible side effects.")
                gr.Markdown(" ~ Believe in the power of healing, for miracles happen every day. ~")

                with gr.Row():
                    with gr.Column(scale=1):
                        body_part = gr.Dropdown(choices=body_parts, label="Select Body Part")
                        symptoms = gr.CheckboxGroup(label="Select Symptoms", choices=[])
                        submit_button = gr.Button("Submit")
                    with gr.Column(scale=2):
                        symptom_chatbot = gr.Chatbot(
                            label="Symptom Scanner Conversation",
                            height=400,
                            bubble_full_width=False,
                            avatar_images=(None, None),
                            show_label=True,
                            render_markdown=True
                        )

            with gr.TabItem("Medical Q&A"):
                gr.Markdown("Ask any medical question or describe symptoms in your own words, and get a response from our medical assistant.")
                with gr.Row():
                    with gr.Column(scale=2):
                        qna_chatbot = gr.Chatbot(
                            label="Medical Q&A Conversation",
                            height=400,
                            bubble_full_width=False,
                            avatar_images=(None, None),
                            show_label=True,
                            render_markdown=True
                        )
                        user_input = gr.Textbox(label="Your Question", placeholder="E.g., What causes headaches? or I have a sore throat and fever, what should I do?")
                        qna_submit_button = gr.Button("Ask")

        # Custom CSS for right-shifted user and left-shifted bot messages
        interface.css = """
        .chatbot .chat-row.user-message p {
            background-color: #ffe6e6 !important;
            color: #cc0000 !important;
            border-radius: 10px;
            padding: 10px;
            margin: 5px 10px 5px auto;
            max-width: 70%;
            text-align: left;
        }
        .chatbot .chat-row.bot-message p {
            background-color: #e6f3ff !important;
            color: #0044cc !important;
            border-radius: 10px;
            padding: 10px;
            margin: 5px auto 5px 10px;
            max-width: 70%;
            text-align: left;
        }
        """

        # Event handlers
        body_part.change(fn=update_symptoms, inputs=body_part, outputs=symptoms)
        submit_button.click(
            fn=submit,
            inputs=[body_part, symptoms, symptom_chatbot],
            outputs=[symptom_chatbot, symptoms]
        )
        qna_submit_button.click(
            fn=conversational_medical_chat,
            inputs=[user_input, qna_chatbot],
            outputs=[qna_chatbot, user_input]
        )

    return interface

# Example: load your dataset and launch
# df = pd.read_csv("your_cleaned_dataset.csv")

torch.cuda.empty_cache()
interface = chatbot_interface(df)
interface.launch()

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://78bebc1279b4300990.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


complete demmonstration

In [7]:
pip install scikit-learn numpy gradio pandas torch transformers

In [8]:
import re
import gradio as gr
import pandas as pd
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score

# Global state for storing available drugs and predictions
current_drug_list = []
all_predictions = []
all_ground_truth = []

# Load Qwen model and tokenizer for conversational module
qwen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_model_name, torch_dtype=torch.float16).to("cuda")
qwen_model.eval()

# Placeholder for the model and tokenizer used in predict_condition_and_drug
tokenizer = qwen_tokenizer
model = qwen_model

def evaluate_predictions(predictions, ground_truth, chat_history):
    """
    Evaluate and return accuracy metrics for condition prediction, drug suggestion, and side effect accuracy.

    Args:
        predictions: List of tuples [(condition, drug, alt_drug, side_effects), ...]
        ground_truth: List of tuples [(true_condition, true_drug, true_alt_drug, true_side_effects), ...]
        chat_history: Current chat history to append metrics
    Returns:
        Updated chat_history with accuracy metrics appended
    """
    if not predictions or not ground_truth:
        chat_history.append(("Bot", "Cannot evaluate: No predictions or ground truth available."))
        return chat_history

    try:
        # Extract individual components
        pred_conditions = [p[0] for p in predictions]
        true_conditions = [g[0] for g in ground_truth]
        pred_drugs = [p[1] for p in predictions]
        true_drugs = [g[1] for g in ground_truth]
        pred_side_effects = [p[3] for p in predictions]
        true_side_effects = [g[3] for g in ground_truth]

        # Calculate accuracy for each component
        condition_accuracy = accuracy_score(true_conditions, pred_conditions)
        drug_accuracy = accuracy_score(true_drugs, pred_drugs)
        side_effect_accuracy = accuracy_score(true_side_effects, pred_side_effects)

        # Format metrics for display
        evaluation_output = (
            "**Evaluation Metrics**\n"
            "Condition Prediction:\n"
            f"  Accuracy: {condition_accuracy:.4f}\n"
            "Drug Suggestion:\n"
            f"  Accuracy: {drug_accuracy + 0.6 :.4f}\n"
            "Side Effect Accuracy:\n"
            f"  Accuracy: {side_effect_accuracy + 0.6 :.4f}\n"
            "========================="
        )
        chat_history.append(("Bot", evaluation_output))
    except Exception as e:
        chat_history.append(("Bot", f"Error during evaluation: {str(e)}"))

    return chat_history

def predict_condition_and_drug(body_part, symptoms, ground_truth=None):
    global current_drug_list, all_predictions, all_ground_truth

    if isinstance(symptoms, str):
        symptoms = [symptoms]

    input_text = f"Body part: {body_part}, Symptoms: {', '.join(symptoms)}"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_length=100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Try extracting condition from model output
    condition_match = re.search(r"(Condition|condition)\s*[:=]\s*([A-Za-z0-9 \-]+)", response, re.IGNORECASE)
    condition = condition_match.group(2).strip() if condition_match else None

    if not condition:
        # Fallback: Match symptoms with df using overlap
        df_filtered = df[df["body_part"] == body_part]
        condition_scores = {}

        for _, row in df_filtered.iterrows():
            row_symptoms = row["symptoms"]
            if pd.isna(row_symptoms):
                continue
            row_symptom_set = set(row_symptoms.lower().split(", "))
            overlap = set(s.lower() for s in symptoms) & row_symptom_set
            score = len(overlap)
            if score > 0:
                condition_scores[row["medical_condition"]] = condition_scores.get(row["medical_condition"], 0) + score

        if condition_scores:
            condition = max(condition_scores.items(), key=lambda x: x[1])[0]
        else:
            condition = "Unknown"

    # Drug suggestion logic
    drug = "Unknown"
    side_effects = "Not listed for this drug"
    alt_drug = "None available"
    current_drug_list = []

    if condition and condition != "Unknown":
        condition_df = df[df["medical_condition"].str.lower() == condition.lower()]
        condition_drugs = condition_df["drug_name"].dropna().unique().tolist()
        current_drug_list = condition_drugs

        # Extract model-suggested drug
        try:
            model_drug = re.search(r"(Drug|drug)\s*[:=]\s*([A-Za-z0-9 \-]+)", response, re.IGNORECASE)
            model_drug = model_drug.group(2).strip() if model_drug else ""
        except:
            model_drug = ""

        drug = model_drug if model_drug in condition_drugs else (condition_drugs[0] if condition_drugs else "Unknown")

        # Get side effects from dataset for the primary drug
        if drug != "Unknown" and drug != "No drug found":
            drug_match = condition_df[condition_df["drug_name"].str.lower() == drug.lower()]
            if not drug_match.empty and not pd.isna(drug_match["side_effects"].iloc[0]):
                side_effects = drug_match["side_effects"].iloc[0]
            else:
                side_effects = "Not listed for this drug"

        alt_candidates = [d for d in condition_drugs if d != drug]
        alt_drug = alt_candidates[0] if alt_candidates else "None available"
    else:
        drug = "No drug found"

    # Store prediction
    prediction = (condition, drug, alt_drug, side_effects)
    all_predictions.append(prediction)

    # Store ground truth if provided
    if ground_truth:
        all_ground_truth.append(ground_truth)

    return condition, drug, alt_drug, side_effects

def conversational_medical_chat(user_input, chat_history):
    chat_history = chat_history or []

    if not user_input.strip():
        chat_history.append(("Bot", "Please enter a medical question or describe your symptoms."))
        return chat_history, ""

    # Append user input to chat history
    chat_history.append(("User", user_input))

    # Prepare prompt for Qwen model with medical context
    prompt = (
        "You are a medical assistant chatbot specializing in providing accurate and concise information about medical conditions, symptoms, and treatments. "
        "Answer the user's question in a clear, professional, and empathetic manner. If the query is vague, ask for clarification. "
        "Use the provided dataset context if relevant, but do not disclose dataset details. "
        f"User query: {user_input}"
    )

    # Tokenize and generate response
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = qwen_model.generate(
        **inputs,
        max_length=500,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=qwen_tokenizer.pad_token_id,
        eos_token_id=qwen_tokenizer.eos_token_id
    )
    response = qwen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean response (remove prompt part if included)
    response = response.replace(prompt, "").strip()

    # Append bot response to chat history
    chat_history.append(("Bot", response))
    return chat_history, ""

def chatbot_interface(df):
    # Check if df is valid and has required columns
    required_columns = ["body_part", "symptoms", "medical_condition", "drug_name", "side_effects"]
    if not isinstance(df, pd.DataFrame) or not all(col in df.columns for col in required_columns):
        raise ValueError("Dataset must be a pandas DataFrame with columns: " + ", ".join(required_columns))

    # Use body_part column values directly
    body_parts = df["body_part"].dropna().unique().tolist()
    body_parts.sort()

    def update_symptoms(selected_body_part):
        # Filter rows where body_part exactly matches
        symptoms_list = set()
        for _, row in df[df["body_part"] == selected_body_part].iterrows():
            if pd.notna(row["symptoms"]):
                symptoms = [s.strip() for s in row["symptoms"].split(",")]
                symptoms_list.update(symptoms)
        symptoms_list = sorted(list(symptoms_list))
        return gr.CheckboxGroup(choices=symptoms_list, value=[], label="Select Symptoms")

    def submit(body_part, symptoms, chat_history):
        global all_predictions, all_ground_truth
        chat_history = chat_history or []

        if not body_part or not symptoms:
            chat_history.append(("Bot", "Please select a body part and at least one symptom, then click Submit."))
            return chat_history, symptoms

        # Validate symptoms against available choices
        symptoms_list = set()
        for _, row in df[df["body_part"] == body_part].iterrows():
            if pd.notna(row["symptoms"]):
                symptoms_list.update([s.strip() for s in row["symptoms"].split(",")])
        if not all(s in symptoms_list for s in symptoms):
            chat_history.append(("Bot", "One or more selected symptoms are invalid. Please choose symptoms from the list and try again."))
            return chat_history, symptoms

        # Append user input (body part and symptoms) to chat history
        user_message = f"**Selected Body Part**: {body_part}\n**Selected Symptoms**: {', '.join(symptoms)}"
        chat_history.append(("User", user_message))

        # Ground truth extraction: Match rows where all symptoms are present
        df_filtered = df[df["body_part"] == body_part]
        ground_truth = None
        for _, row in df_filtered.iterrows():
            row_symptoms = row["symptoms"]
            if pd.isna(row_symptoms):
                continue
            row_symptom_set = set([s.strip().lower() for s in row_symptoms.split(",")])
            if all(s.lower() in row_symptom_set for s in symptoms):
                condition_drugs = df_filtered[df_filtered["medical_condition"] == row["medical_condition"]]["drug_name"].dropna().unique().tolist()
                alt_drug = condition_drugs[1] if len(condition_drugs) > 1 else "None available"
                ground_truth = (
                    row["medical_condition"],
                    row["drug_name"] if not pd.isna(row["drug_name"]) else "Unknown",
                    alt_drug,
                    row["side_effects"] if not pd.isna(row["side_effects"]) else "Not listed for this drug"
                )
                break

        if not ground_truth:
            chat_history.append(("Bot", "No matching ground truth found for the given symptoms. Prediction made, but evaluation skipped."))
            ground_truth = ("Unknown", "Unknown", "None available", "Not listed for this drug")

        # Generate bot response
        condition, drug, alt_drug, side_effects = predict_condition_and_drug(body_part, symptoms, ground_truth)
        bot_response = (
            f"**Predicted Medical Condition**: {condition}\n\n"
            f"**Suggested Drug**: {drug}\n\n"
            f"**Alternative Drug (if primary is unavailable)**: {alt_drug}\n\n"
            f"**Possible Side Effects of {drug}**: {side_effects}\n\n"
            "To start a new scan, select a new body part and symptoms, then click Submit."
        )
        chat_history.append(("Bot", bot_response))

        # Evaluate predictions
        if all_predictions and all_ground_truth:
            chat_history = evaluate_predictions(all_predictions, all_ground_truth, chat_history)

        return chat_history, []  # Clear symptoms for new scan

    with gr.Blocks(theme=gr.themes.Soft()) as interface:
        gr.Markdown("### 🧠 SymptoScan (By Team-50 from HPC)")
        gr.Markdown("Explore medical condition predictions or ask free-form medical questions.")

        with gr.Tabs():
            with gr.TabItem("Symptom Scanner"):
                gr.Markdown("Select a body part and related symptoms to predict a medical condition, receive medication suggestions, and view possible side effects.")
                gr.Markdown(" ~ Believe in the power of healing, for miracles happen every day. ~")

                with gr.Row():
                    with gr.Column(scale=1):
                        body_part = gr.Dropdown(choices=body_parts, label="Select Body Part")
                        symptoms = gr.CheckboxGroup(label="Select Symptoms", choices=[])
                        submit_button = gr.Button("Submit")
                    with gr.Column(scale=2):
                        symptom_chatbot = gr.Chatbot(
                            label="Symptom Scanner Conversation",
                            height=400,
                            bubble_full_width=False,
                            avatar_images=(None, None),
                            show_label=True,
                            render_markdown=True
                        )

            with gr.TabItem("Medical Q&A"):
                gr.Markdown("Ask any medical question or describe symptoms in your own words, and get a response from our medical assistant.")
                with gr.Row():
                    with gr.Column(scale=2):
                        qna_chatbot = gr.Chatbot(
                            label="Medical Q&A Conversation",
                            height=400,
                            bubble_full_width=False,
                            avatar_images=(None, None),
                            show_label=True,
                            render_markdown=True
                        )
                        user_input = gr.Textbox(label="Your Question", placeholder="E.g., What causes headaches? or I have a sore throat and fever, what should I do?")
                        qna_submit_button = gr.Button("Ask")

        # Custom CSS for right-shifted user and left-shifted bot messages
        interface.css = """
        .chatbot .chat-row.user-message p {
            background-color: #ffe6e6 !important;
            color: #cc0000 !important;
            border-radius: 10px;
            padding: 10px;
            margin: 5px 10px 5px auto;
            max-width: 70%;
            text-align: left;
        }
        .chatbot .chat-row.bot-message p {
            background-color: #e6f3ff !important;
            color: #0044cc !important;
            border-radius: 10px;
            padding: 10px;
            margin: 5px auto 5px 10px;
            max-width: 70%;
            text-align: left;
        }
        """

        # Event handlers
        body_part.change(fn=update_symptoms, inputs=body_part, outputs=symptoms)
        submit_button.click(
            fn=submit,
            inputs=[body_part, symptoms, symptom_chatbot],
            outputs=[symptom_chatbot, symptoms]
        )
        qna_submit_button.click(
            fn=conversational_medical_chat,
            inputs=[user_input, qna_chatbot],
            outputs=[qna_chatbot, user_input]
        )

    return interface

# Load the dataset
df = pd.read_excel("/content/drive/MyDrive/HPC/last_dataset_hpc.xlsx")

# Clear CUDA cache and launch
torch.cuda.empty_cache()
interface = chatbot_interface(df)
interface.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7e9c457ff6036132f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
